# DSMarket — Tarea 5: Pipeline y API para forecasting operativo

<div style="background-color:#2D8C4E;border-left:6px solid #2D8C4E;padding:14px;border-radius:10px">

**Objetivo del notebook**

Demostrar cómo trasladar la solución final del proyecto a una forma operativa mínima y consumible por otros sistemas.

Este notebook no entrena de nuevo el modelo.  
Su propósito es mostrar:

- cómo preparar los datos de entrada,
- cómo cargar el modelo final recomendado,
- cómo aplicar la calibración operativa,
- y cómo exponer una recomendación de pedido mediante una API sencilla.

</div>

<a id="indice"></a>

## Índice

1. [Qué demuestra este notebook](#que-demuestra)
2. [Setup e inputs necesarios](#setup)
3. [Carga de artefactos y configuración](#artefactos)
4. [Pipeline mínimo de preparación](#pipeline)
5. [Aplicación del modelo y calibración](#modelo-calibracion)
6. [Diseño de la API](#api)
7. [Demostración de inferencia](#inferencia)
8. [Conclusión ejecutiva](#cierre)

<a id="que-demuestra"></a>

## Qué demuestra este notebook

Este notebook representa la transición desde el análisis en notebook a una solución operativa mínima.

La lógica es la siguiente:

1. cargar los datos y artefactos necesarios,
2. preparar las variables de entrada con un pipeline reproducible,
3. aplicar el modelo final recomendado,
4. corregir la predicción con la calibración validada en la Tarea 3,
5. y mostrar cómo esa salida podría exponerse mediante API.

La solución operativa que se toma como referencia en este notebook es:

- **modelo técnico ganador:** E2
- **solución operativa recomendada:** E2 calibrado

[⬆ Volver al índice](#indice)

In [3]:
# =============================================================================
# Imports y configuración global
# =============================================================================

import os
import re
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
np.random.seed(SEED)

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

COLORS = {
    "primary":   "#2D8C4E",
    "secondary": "#F4A261",
    "accent":    "#E76F51",
    "neutral":   "#8ECAE6",
    "dark":      "#264653",
}

CALIBRATION_FACTOR = 1.3657

print(f"✅ Configuración cargada | SEED={SEED}")
print(f"✅ Factor de calibración operativo: {CALIBRATION_FACTOR}")


✅ Configuración cargada | SEED=42
✅ Factor de calibración operativo: 1.3657


<a id="setup"></a>

## Setup e inputs necesarios

Este notebook necesita distinguir entre dos tipos de inputs:

### Datos base del proyecto
- `daily_calendar_with_events.csv`
- `item_prices.csv`
- `item_sales.csv`

### Artefactos del modelo final
- `model_E2.txt`
- factor de calibración de E2

### Decisión de alcance

En esta tarea la solución operativa se apoya en **E2 calibrado**.  
Por tanto:

- **no es obligatorio cargar clusters de producto**
- **no es necesario reentrenar modelos**
- **no hace falta volver a ejecutar el forecasting completo**

[⬆ Volver al índice](#indice)

In [6]:
# =============================================================================
# Rutas portables del proyecto
# =============================================================================

PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR_DEFAULT = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
MODELS_DIR = OUTPUTS_DIR / "forecasting_models"

for path in [INTERIM_DIR, PROCESSED_DIR, FIGURES_DIR, OUTPUTS_DIR, MODELS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

REQUIRED_RAW_FILES = {
    "daily_calendar_with_events.csv",
    "item_prices.csv",
    "item_sales.csv",
}

def has_required_files(folder: Path) -> bool:
    return folder.is_dir() and REQUIRED_RAW_FILES.issubset({p.name for p in folder.iterdir() if p.is_file()})

def build_candidate_raw_dirs() -> list[Path]:
    candidates = []

    env_raw = os.getenv("DSMARKET_RAW_DIR")
    if env_raw:
        candidates.append(Path(env_raw))

    candidates.extend([
        PROJECT_ROOT / "data" / "raw",
        PROJECT_ROOT / "raw",
        PROJECT_ROOT.parent / "data" / "raw",
        Path("/content/DSMarket/data/raw"),
        Path("/content/drive/MyDrive/TFM MASTER/TFM/Mateo/data/raw"),
        Path("/kaggle/input/dsmarket"),
        Path("/kaggle/input/datasets/mateopascual/dsmarket"),
    ])

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for path in kaggle_input.iterdir():
            if path.is_dir():
                candidates.append(path)
                candidates.extend([sub for sub in path.iterdir() if sub.is_dir()])

    unique_candidates = []
    seen = set()

    for path in candidates:
        path = Path(path)
        if str(path) not in seen:
            unique_candidates.append(path)
            seen.add(str(path))

    return unique_candidates

def resolve_raw_dir() -> Path:
    candidates = build_candidate_raw_dirs()

    for candidate in candidates:
        if has_required_files(candidate):
            return candidate
        if has_required_files(candidate / "data" / "raw"):
            return candidate / "data" / "raw"
        if has_required_files(candidate / "raw"):
            return candidate / "raw"

    searched = "\n - ".join(str(p) for p in candidates[:20])
    raise FileNotFoundError(
        "No se encontró una carpeta válida con los archivos requeridos.\n"
        "Archivos esperados:\n"
        f" - {chr(10).join(sorted(REQUIRED_RAW_FILES))}\n\n"
        "Puedes definir manualmente la variable de entorno DSMARKET_RAW_DIR.\n"
        f"Primeras rutas revisadas:\n - {searched}"
    )

RAW_DIR = resolve_raw_dir()

print("✅ RAW_DIR resuelto:", RAW_DIR)
print("✅ MODELS_DIR:", MODELS_DIR)

✅ RAW_DIR resuelto: /kaggle/input/datasets/mateopascual/dsmarket
✅ MODELS_DIR: /kaggle/working/outputs/forecasting_models


In [11]:
# =============================================================================
# Detección de artefactos del modelo final
# =============================================================================

def resolve_model_e2_path() -> Path | None:
    candidates = [
        MODELS_DIR / "model_E2.txt",
        OUTPUTS_DIR / "forecasting_models" / "model_E2.txt",
        PROJECT_ROOT / "models" / "model_E2.txt",
        PROJECT_ROOT / "outputs" / "forecasting_models" / "model_E2.txt",
        RAW_DIR / "model_E2.txt",
    ]

    for candidate in candidates:
        if candidate.is_file():
            return candidate

    return None

MODEL_E2_PATH = resolve_model_e2_path()

if MODEL_E2_PATH is None:
    print("ℹ️ No se encontró model_E2.txt todavía.")
    print("ℹ️ El notebook podrá dejar preparado el pipeline y la API,")
    print("ℹ️ pero la inferencia real con el modelo quedará pendiente hasta cargar ese artefacto.")
else:
    print("✅ Modelo E2 encontrado en:", MODEL_E2_PATH)

✅ Modelo E2 encontrado en: /kaggle/input/datasets/mateopascual/dsmarket/model_E2.txt


In [12]:
# =============================================================================
# Carga de datos base
# =============================================================================

sales_dtypes = {
    "item": "category",
    "store_code": "category",
    "category": "category",
}

prices_dtypes = {
    "item": "category",
    "store_code": "category",
}

df_sales = pd.read_csv(
    RAW_DIR / "item_sales.csv",
    dtype=sales_dtypes,
    low_memory=False
)

df_prices = pd.read_csv(
    RAW_DIR / "item_prices.csv",
    dtype=prices_dtypes,
    low_memory=False
)

df_cal = pd.read_csv(
    RAW_DIR / "daily_calendar_with_events.csv",
    parse_dates=["date"],
    low_memory=False
)

print("✅ Datos base cargados")
print("df_sales :", df_sales.shape)
print("df_prices:", df_prices.shape)
print("df_cal   :", df_cal.shape)

✅ Datos base cargados
df_sales : (30490, 1920)
df_prices: (6965706, 5)
df_cal   : (1913, 5)


In [13]:
# =============================================================================
# Resumen de inputs disponibles
# =============================================================================

inputs_status = pd.DataFrame([
    {"input": "daily_calendar_with_events.csv", "disponible": True},
    {"input": "item_prices.csv",                "disponible": True},
    {"input": "item_sales.csv",                 "disponible": True},
    {"input": "model_E2.txt",                   "disponible": MODEL_E2_PATH is not None},
    {"input": "factor_calibracion",             "disponible": True},
])

display(inputs_status)

,input,disponible
0,daily_calendar_with_events.csv,True
1,item_prices.csv,True
2,item_sales.csv,True
3,model_E2.txt,True
4,factor_calibracion,True


<a id="artefactos"></a>

## Carga de artefactos y configuración

En esta sección cargo el modelo final recomendado y dejo definidos los elementos mínimos necesarios para la inferencia.

La lógica operativa de referencia será:

- **modelo base:** E2
- **corrección operativa:** calibración global
- **salida final:** forecast calibrado y recomendación de pedido

[⬆ Volver al índice](#indice)

In [14]:
# =============================================================================
# Carga del modelo final E2
# =============================================================================

model_E2 = None

if MODEL_E2_PATH is not None:
    model_E2 = lgb.Booster(model_file=str(MODEL_E2_PATH))
    print("✅ model_E2 cargado correctamente")
    print("Ruta:", MODEL_E2_PATH)
else:
    print("ℹ️ No se pudo cargar model_E2 porque no se encontró model_E2.txt")

✅ model_E2 cargado correctamente
Ruta: /kaggle/input/datasets/mateopascual/dsmarket/model_E2.txt


In [15]:
# =============================================================================
# Features operativas del modelo E2
# =============================================================================

FEATURES_E2_API = [
    "lag_7",
    "lag_14",
    "lag_21",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_28",
    "day_of_week",
    "day_of_month",
    "week_of_year",
    "month",
    "year",
    "is_weekend",
    "has_event",
]

print("✅ Features operativas definidas")
print(FEATURES_E2_API)

✅ Features operativas definidas
['lag_7', 'lag_14', 'lag_21', 'lag_28', 'rolling_mean_7', 'rolling_mean_28', 'day_of_week', 'day_of_month', 'week_of_year', 'month', 'year', 'is_weekend', 'has_event']


<a id="pipeline"></a>

## Pipeline mínimo de preparación

El objetivo aquí no es reconstruir el histórico completo del notebook 3,
sino demostrar cómo se prepararía una entrada operativa para inferencia.

La lógica del pipeline mínimo será:

1. seleccionar una combinación tienda × producto,
2. recuperar su historia reciente,
3. construir las features necesarias,
4. aplicar el modelo,
5. calibrar la salida,
6. y convertirla en una recomendación de pedido.

[⬆ Volver al índice](#indice)

In [16]:
# =============================================================================
# Preparación mínima del panel diario
# =============================================================================

day_cols = [c for c in df_sales.columns if re.fullmatch(r"d_\d+", str(c))]
day_cols = sorted(day_cols, key=lambda x: int(str(x).split("_")[1]))

meta_cols = ["item", "store_code", "category"]

df_panel_api = df_sales[meta_cols + day_cols].melt(
    id_vars=meta_cols,
    value_vars=day_cols,
    var_name="d",
    value_name="units"
)

df_cal_slim = df_cal[["d", "date"]].copy()
df_panel_api = df_panel_api.merge(df_cal_slim, on="d", how="left")

df_panel_api["date"] = pd.to_datetime(df_panel_api["date"], errors="coerce")
df_panel_api["units"] = pd.to_numeric(df_panel_api["units"], errors="coerce").fillna(0).astype("float32")

df_panel_api = df_panel_api.sort_values(["store_code", "item", "date"]).reset_index(drop=True)

print("✅ df_panel_api preparado")
print(df_panel_api.shape)
display(df_panel_api.head(5))

✅ df_panel_api preparado
(58327370, 6)


,item,store_code,category,d,units,date
0,ACCESORIES_1_001,BOS_1,ACCESORIES,d_1,0.0000,2011-01-29
1,ACCESORIES_1_001,BOS_1,ACCESORIES,d_2,0.0000,2011-01-30
2,ACCESORIES_1_001,BOS_1,ACCESORIES,d_3,0.0000,2011-01-31
3,ACCESORIES_1_001,BOS_1,ACCESORIES,d_4,0.0000,2011-02-01
4,ACCESORIES_1_001,BOS_1,ACCESORIES,d_5,0.0000,2011-02-02


In [17]:
# =============================================================================
# Construcción mínima de features para inferencia
# =============================================================================

df_panel_api["day_of_week"] = df_panel_api["date"].dt.dayofweek.astype("int8")
df_panel_api["day_of_month"] = df_panel_api["date"].dt.day.astype("int8")
df_panel_api["week_of_year"] = df_panel_api["date"].dt.isocalendar().week.astype("int16")
df_panel_api["month"] = df_panel_api["date"].dt.month.astype("int8")
df_panel_api["year"] = df_panel_api["date"].dt.year.astype("int16")
df_panel_api["is_weekend"] = (df_panel_api["day_of_week"] >= 5).astype("int8")

for lag in [7, 14, 21, 28]:
    df_panel_api[f"lag_{lag}"] = (
        df_panel_api.groupby(["store_code", "item"], observed=True)["units"]
        .shift(lag)
        .astype("float32")
    )

for window in [7, 28]:
    df_panel_api[f"rolling_mean_{window}"] = (
        df_panel_api.groupby(["store_code", "item"], observed=True)["units"]
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
        .astype("float32")
    )

if "event" in df_cal.columns:
    df_event = df_cal[["date", "event"]].copy()
    df_event["has_event"] = df_event["event"].notna().astype("int8")
else:
    df_event = df_cal[["date"]].copy()
    df_event["has_event"] = 0

df_panel_api = df_panel_api.drop(columns=["has_event"], errors="ignore")
df_panel_api = df_panel_api.merge(df_event[["date", "has_event"]], on="date", how="left")
df_panel_api["has_event"] = df_panel_api["has_event"].fillna(0).astype("int8")

print("✅ Features mínimas para API construidas")
display(df_panel_api[["store_code", "item", "date"] + FEATURES_E2_API].head(5))

✅ Features mínimas para API construidas


,store_code,item,date,lag_7,lag_14,lag_21,lag_28,rolling_mean_7,rolling_mean_28,day_of_week,day_of_month,week_of_year,month,year,is_weekend,has_event
0,BOS_1,ACCESORIES_1_001,2011-01-29,NaN,NaN,NaN,NaN,NaN,NaN,5,29,4,1,2011,1,0
1,BOS_1,ACCESORIES_1_001,2011-01-30,NaN,NaN,NaN,NaN,0.0000,0.0000,6,30,4,1,2011,1,0
2,BOS_1,ACCESORIES_1_001,2011-01-31,NaN,NaN,NaN,NaN,0.0000,0.0000,0,31,5,1,2011,0,0
3,BOS_1,ACCESORIES_1_001,2011-02-01,NaN,NaN,NaN,NaN,0.0000,0.0000,1,1,5,2,2011,0,0
4,BOS_1,ACCESORIES_1_001,2011-02-02,NaN,NaN,NaN,NaN,0.0000,0.0000,2,2,5,2,2011,0,0


In [18]:
# =============================================================================
# Snapshot más reciente por tienda × producto
# =============================================================================

df_latest_snapshot = (
    df_panel_api
    .sort_values(["store_code", "item", "date"])
    .groupby(["store_code", "item"], observed=True, as_index=False)
    .tail(1)
    .reset_index(drop=True)
)

print("✅ Último snapshot creado")
print(df_latest_snapshot.shape)
display(df_latest_snapshot[["store_code", "item", "category", "date"] + FEATURES_E2_API].head(10))

✅ Último snapshot creado
(30490, 19)


,store_code,item,category,date,lag_7,lag_14,lag_21,lag_28,rolling_mean_7,rolling_mean_28,day_of_week,day_of_month,week_of_year,month,year,is_weekend,has_event
0,BOS_1,ACCESORIES_1_001,ACCESORIES,2016-04-24,0.0000,0.0000,0.0000,0.0000,0.2857,0.2857,6,24,16,4,2016,1,0
1,BOS_1,ACCESORIES_1_002,ACCESORIES,2016-04-24,0.0000,0.0000,0.0000,0.0000,0.0000,0.0357,6,24,16,4,2016,1,0
2,BOS_1,ACCESORIES_1_003,ACCESORIES,2016-04-24,0.0000,1.0000,0.0000,0.0000,0.0000,0.2143,6,24,16,4,2016,1,0
3,BOS_1,ACCESORIES_1_004,ACCESORIES,2016-04-24,2.0000,2.0000,3.0000,2.0000,0.7143,0.8214,6,24,16,4,2016,1,0
4,BOS_1,ACCESORIES_1_005,ACCESORIES,2016-04-24,2.0000,2.0000,0.0000,0.0000,1.7143,0.8929,6,24,16,4,2016,1,0
5,BOS_1,ACCESORIES_1_006,ACCESORIES,2016-04-24,1.0000,2.0000,0.0000,0.0000,0.2857,0.4643,6,24,16,4,2016,1,0
6,BOS_1,ACCESORIES_1_007,ACCESORIES,2016-04-24,0.0000,0.0000,0.0000,2.0000,0.2857,0.3214,6,24,16,4,2016,1,0
7,BOS_1,ACCESORIES_1_008,ACCESORIES,2016-04-24,0.0000,0.0000,0.0000,4.0000,1.4286,2.2143,6,24,16,4,2016,1,0
8,BOS_1,ACCESORIES_1_009,ACCESORIES,2016-04-24,0.0000,0.0000,0.0000,0.0000,0.4286,0.3929,6,24,16,4,2016,1,0
9,BOS_1,ACCESORIES_1_010,ACCESORIES,2016-04-24,1.0000,0.0000,1.0000,4.0000,0.4286,0.3929,6,24,16,4,2016,1,0


In [20]:
# =============================================================================
# Validación de columnas necesarias para inferencia
# =============================================================================

missing_cols = [c for c in FEATURES_E2_API if c not in df_latest_snapshot.columns]

if missing_cols:
    raise ValueError(f"Faltan columnas necesarias para inferencia: {missing_cols}")

null_summary = df_latest_snapshot[FEATURES_E2_API].isna().sum()

print("✅ Validación de inferencia completada")
print("\nNaNs por feature:")
display(null_summary.to_frame("n_nulls").T)

✅ Validación de inferencia completada

NaNs por feature:


,lag_7,lag_14,lag_21,lag_28,rolling_mean_7,rolling_mean_28,day_of_week,day_of_month,week_of_year,month,year,is_weekend,has_event
n_nulls,0,0,0,0,0,0,0,0,0,0,0,0,0


In [22]:
# =============================================================================
# Inferencia con model_E2
# =============================================================================

if model_E2 is None:
    print("ℹ️ No se puede hacer inferencia real porque no está cargado model_E2.txt")
    df_inference = df_latest_snapshot.copy()
    df_inference["pred_E2_raw"] = np.nan
else:
    X_api = df_latest_snapshot[FEATURES_E2_API].values.astype("float32")

    df_inference = df_latest_snapshot.copy()
    df_inference["pred_E2_raw"] = np.clip(model_E2.predict(X_api), 0, None).astype("float32")

    print("✅ Inferencia E2 completada")
    display(
        df_inference[
            ["store_code", "item", "category", "date", "pred_E2_raw"]
        ].head(10)
    )

✅ Inferencia E2 completada


,store_code,item,category,date,pred_E2_raw
0,BOS_1,ACCESORIES_1_001,ACCESORIES,2016-04-24,0.0000
1,BOS_1,ACCESORIES_1_002,ACCESORIES,2016-04-24,0.0000
2,BOS_1,ACCESORIES_1_003,ACCESORIES,2016-04-24,0.0000
3,BOS_1,ACCESORIES_1_004,ACCESORIES,2016-04-24,1.0025
4,BOS_1,ACCESORIES_1_005,ACCESORIES,2016-04-24,1.0049
5,BOS_1,ACCESORIES_1_006,ACCESORIES,2016-04-24,0.0001
6,BOS_1,ACCESORIES_1_007,ACCESORIES,2016-04-24,0.0000
7,BOS_1,ACCESORIES_1_008,ACCESORIES,2016-04-24,1.7621
8,BOS_1,ACCESORIES_1_009,ACCESORIES,2016-04-24,0.0000
9,BOS_1,ACCESORIES_1_010,ACCESORIES,2016-04-24,0.0080


<a id="modelo-calibracion"></a>

## Aplicación del modelo y calibración

En esta sección aplico el modelo final a un snapshot reciente de cada combinación tienda × producto.

La lógica operativa es:

1. generar una predicción base con **E2**,
2. corregirla con el factor de calibración validado en la Tarea 3,
3. y dejar preparada una salida que ya pueda consumirse como recomendación operativa.

[⬆ Volver al índice](#indice)

In [23]:
# =============================================================================
# Aplicación de calibración operativa
# =============================================================================

df_inference = df_inference.copy()

df_inference["pred_E2_calibrated"] = (
    df_inference["pred_E2_raw"] * CALIBRATION_FACTOR
).astype("float32")

print("✅ Calibración aplicada")
display(
    df_inference[
        ["store_code", "item", "pred_E2_raw", "pred_E2_calibrated"]
    ].head(10)
)

✅ Calibración aplicada


,store_code,item,pred_E2_raw,pred_E2_calibrated
0,BOS_1,ACCESORIES_1_001,0.0000,0.0000
1,BOS_1,ACCESORIES_1_002,0.0000,0.0000
2,BOS_1,ACCESORIES_1_003,0.0000,0.0000
3,BOS_1,ACCESORIES_1_004,1.0025,1.3691
4,BOS_1,ACCESORIES_1_005,1.0049,1.3724
5,BOS_1,ACCESORIES_1_006,0.0001,0.0002
6,BOS_1,ACCESORIES_1_007,0.0000,0.0000
7,BOS_1,ACCESORIES_1_008,1.7621,2.4065
8,BOS_1,ACCESORIES_1_009,0.0000,0.0000
9,BOS_1,ACCESORIES_1_010,0.0080,0.0109


In [24]:
# =============================================================================
# Regla mínima de recomendación de pedido
# =============================================================================

# Demo simple: suponer stock disponible fijo y safety stock proporcional
DEFAULT_STOCK_ON_HAND = 20.0
SAFETY_STOCK_FACTOR = 0.25

df_inference["stock_on_hand_demo"] = DEFAULT_STOCK_ON_HAND
df_inference["safety_stock_demo"] = (
    df_inference["pred_E2_calibrated"] * SAFETY_STOCK_FACTOR
).astype("float32")

df_inference["recommended_order_demo"] = np.maximum(
    0,
    df_inference["pred_E2_calibrated"] + df_inference["safety_stock_demo"] - df_inference["stock_on_hand_demo"]
).astype("float32")

print("✅ Recomendación operativa demo calculada")
display(
    df_inference[
        [
            "store_code",
            "item",
            "pred_E2_calibrated",
            "stock_on_hand_demo",
            "safety_stock_demo",
            "recommended_order_demo"
        ]
    ].head(10)
)

✅ Recomendación operativa demo calculada


,store_code,item,pred_E2_calibrated,stock_on_hand_demo,safety_stock_demo,recommended_order_demo
0,BOS_1,ACCESORIES_1_001,0.0000,20.0000,0.0000,0.0000
1,BOS_1,ACCESORIES_1_002,0.0000,20.0000,0.0000,0.0000
2,BOS_1,ACCESORIES_1_003,0.0000,20.0000,0.0000,0.0000
3,BOS_1,ACCESORIES_1_004,1.3691,20.0000,0.3423,0.0000
4,BOS_1,ACCESORIES_1_005,1.3724,20.0000,0.3431,0.0000
5,BOS_1,ACCESORIES_1_006,0.0002,20.0000,0.0000,0.0000
6,BOS_1,ACCESORIES_1_007,0.0000,20.0000,0.0000,0.0000
7,BOS_1,ACCESORIES_1_008,2.4065,20.0000,0.6016,0.0000
8,BOS_1,ACCESORIES_1_009,0.0000,20.0000,0.0000,0.0000
9,BOS_1,ACCESORIES_1_010,0.0109,20.0000,0.0027,0.0000


<a id="api"></a>

## Diseño de la API

El objetivo de esta parte no es desplegar un servidor real dentro del notebook,
sino mostrar qué estructura tendría una API capaz de devolver una recomendación de pedido
basada en el modelo final del proyecto.

[⬆ Volver al índice](#indice)

### Request de ejemplo

```json
{
  "store_id": "NYC_3",
  "item_id": "PROD_00142",
  "horizon_days": 7,
  "service_level": 0.95,
  "stock_on_hand": 20
}

In [27]:
# =============================================================================
# Selección de un caso demo para respuesta API
# =============================================================================

demo_row = df_inference.iloc[0].copy()

demo_request = {
    "store_id": str(demo_row["store_code"]),
    "item_id": str(demo_row["item"]),
    "horizon_days": 7,
    "service_level": 0.95,
    "stock_on_hand": float(demo_row["stock_on_hand_demo"]),
}

print("✅ Request demo preparado")
print(json.dumps(demo_request, indent=2))

✅ Request demo preparado
{
  "store_id": "BOS_1",
  "item_id": "ACCESORIES_1_001",
  "horizon_days": 7,
  "service_level": 0.95,
  "stock_on_hand": 20.0
}


In [28]:
# =============================================================================
# Construcción de respuesta tipo API
# =============================================================================

demo_response = {
    "store_id": str(demo_row["store_code"]),
    "item_id": str(demo_row["item"]),
    "forecast_units_1d_raw": round(float(demo_row["pred_E2_raw"]), 2) if pd.notna(demo_row["pred_E2_raw"]) else None,
    "forecast_units_1d_calibrated": round(float(demo_row["pred_E2_calibrated"]), 2) if pd.notna(demo_row["pred_E2_calibrated"]) else None,
    "safety_stock": round(float(demo_row["safety_stock_demo"]), 2),
    "recommended_order": round(float(demo_row["recommended_order_demo"]), 2),
    "model_version": "E2_calibrated_v1",
    "service_level": 0.95,
    "confidence": "medium"
}

print("✅ Response demo construida")
print(json.dumps(demo_response, indent=2))

✅ Response demo construida
{
  "store_id": "BOS_1",
  "item_id": "ACCESORIES_1_001",
  "forecast_units_1d_raw": 0.0,
  "forecast_units_1d_calibrated": 0.0,
  "safety_stock": 0.0,
  "recommended_order": 0.0,
  "model_version": "E2_calibrated_v1",
  "service_level": 0.95,
  "confidence": "medium"
}


### Response de ejemplo

```json
{
  "store_id": "NYC_3",
  "item_id": "PROD_00142",
  "forecast_units_1d_raw": 112.4,
  "forecast_units_1d_calibrated": 153.5,
  "safety_stock": 38.4,
  "recommended_order": 171.9,
  "model_version": "E2_calibrated_v1",
  "service_level": 0.95,
  "confidence": "medium"
}


---

### Celda 27 — Nota metodológica importante

```markdown id="43s6q0"
📌 **Conclusión / Decisión**

Esta demo no pretende representar un sistema productivo completo,
sino demostrar cómo la solución final del proyecto puede traducirse a una inferencia operativa mínima.

La secuencia relevante queda ya clara:

- preparación de features,
- aplicación del modelo E2,
- calibración de la predicción,
- y construcción de una recomendación consumible por API.

[⬆ Volver al índice](#indice)

<a id="cierre"></a>

## Conclusión ejecutiva

En esta última sección sintetizo qué demuestra realmente este notebook
y cuál es su papel dentro del proyecto DSMarket.

[⬆ Volver al índice](#indice)

### 8.1 Qué demuestra esta tarea

La Tarea 5 no introduce un nuevo modelo ni un nuevo experimento de forecasting.

Su aportación consiste en demostrar que la solución final del proyecto
puede trasladarse a una forma operativa mínima y consumible por otros sistemas.

En concreto, este notebook muestra que es posible:

- preparar entradas de inferencia de forma reproducible,
- cargar el modelo final recomendado,
- aplicar la calibración validada en la Tarea 3,
- y construir una respuesta estructurada compatible con una API.

La solución operativa que se toma como referencia es:

- **modelo técnico base:** E2
- **solución operativa recomendada:** E2 calibrado

[⬆ Volver al índice](#indice)

### 8.2 Alcance real de la propuesta

Este notebook debe interpretarse como una **prueba de operacionalización**,
no como una plataforma MLOps completa.

Lo que sí queda demostrado es la lógica mínima necesaria para pasar de análisis a servicio:

1. **pipeline de preparación**
2. **carga del modelo**
3. **aplicación de calibración**
4. **cálculo de recomendación operativa**
5. **estructura de request/response API**

Lo que no se implementa aquí de forma completa es:

- orquestación real de producción,
- autenticación y seguridad de la API,
- despliegue cloud,
- monitorización automática en tiempo real,
- ni gestión completa de inventario y stock disponible.

Esto es consistente con el alcance del TFM:
mostrar una solución técnica creíble y conectada con negocio,
sin sobredimensionar el nivel de implementación real.

[⬆ Volver al índice](#indice)

### 8.3 Valor para negocio y tecnología

Desde el punto de vista de negocio, este notebook demuestra que la predicción ya puede convertirse en una recomendación operativa interpretable.

Desde el punto de vista tecnológico, demuestra que la solución puede exponerse de forma estructurada mediante API,
lo que facilita su integración futura con:

- ERP,
- herramientas de operaciones,
- dashboards,
- o procesos automáticos de reposición.

En otras palabras, la Tarea 5 conecta el resultado analítico del proyecto
con una arquitectura mínima de uso real.

[⬆ Volver al índice](#indice)

## Memorando ejecutivo — Respuesta a la Tarea 5

<div style="background-color:#2D8C4E;border-left:6px solid #264653;padding:14px;border-radius:10px">

**Asunto:** Pipeline y API para operacionalizar la solución de forecasting de DSMarket

La conclusión de esta tarea es que la solución final del proyecto
ya puede traducirse a una forma operativa mínima y consumible por otros sistemas.

La arquitectura propuesta se apoya en cuatro elementos:

- preparación reproducible de features,
- carga del modelo final recomendado,
- aplicación de calibración operativa,
- y exposición del resultado mediante una API sencilla.

Esto permite que la predicción deje de vivir solo en notebook
y pase a convertirse en una recomendación de pedido integrable con procesos internos de negocio.

La solución presentada no pretende ser una plataforma de producción completa,
pero sí demuestra una transición realista desde el análisis al servicio.

En consecuencia, la recomendación es utilizar esta arquitectura como base técnica
para el piloto propuesto en la Tarea 4
y evolucionarla posteriormente con monitorización, recalibración periódica y despliegue controlado.

</div>

[⬆ Volver al índice](#indice)

---

*Documento elaborado por Nicole Chen, Data Scientist Senior — DSMarket*  
*Marzo 2025*